# 🧠 AI Personal Knowledge Assistant project

A website-style AI assistant that runs fully inside **Google Colab**.
Upload PDF / DOCX / TXT files, then ask questions, make study notes, take quizzes,
research the web and compare documents. Everything is powered by a **LangGraph
Main Coordinator Agent** and five specialised agents, using **RAG (FAISS + Sentence Transformers)**
and any **OpenAI-compatible LLM**.

## Beginner setup
1. Open Google Colab → **File → Upload notebook** (or paste each cell in order).
2. *(Optional, safest)* Add your key in Colab: 🔑 **Secrets** → name `LLM_API_KEY` → paste key → enable notebook access.
3. Run the cells **top to bottom**. In Cell 4 enter your API key, Base URL and Model name.
4. The last cell prints a **public Gradio link**. Open it in your browser.
5. The link works only while the Colab runtime is running. It expires when the runtime stops.
   Documents and chat history are stored in the session only, so they are lost when Colab disconnects.

## Architecture diagram
```mermaid
flowchart TD
    U[User] --> UI[Website UI - Gradio]
    UI --> C[Main Coordinator Agent - LangGraph]
    C --> R{Intelligent Routing}
    R --> K[Knowledge Agent]
    R --> S[Study Agent]
    R --> Q[Quiz Agent]
    R --> W[Research Agent]
    R --> D[Document Comparison Agent]
    K --> T1[RAG / FAISS / Document Processing]
    S --> T1
    Q --> T1
    D --> T1
    W --> T2[Web Search - DuckDuckGo]
    T1 --> L[LLM]
    T2 --> L
    L --> F[Final Response with Sources]
    F --> UI
```

## RAG workflow diagram
```mermaid
flowchart TD
    A[Document Upload] --> B[Document Extraction]
    B --> C[Chunking]
    C --> D[Embedding Generation]
    D --> E[FAISS Storage]
    F[User Query] --> G[Query Embedding]
    G --> H[Similarity Search]
    E --> H
    H --> I[Relevant Context]
    I --> J[LLM]
    J --> K[Cited Answer]
```

In [1]:
!pip install -q "gradio>=5.0,<6.0" langgraph langchain-core faiss-cpu sentence-transformers pymupdf python-docx ddgs openai plotly pandas numpy requests
print("✅ Dependencies installed. If Colab asks to restart the runtime, restart it and continue from Cell 3.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 2.12.1 requires pydantic<3.0.0,>=2.12.5, but you have pydantic 2.12.3 which is incompatible.
hf-gradio 0.4.1 requires gradio-client<3.0,>=2.0, but you have gradio-client 1.14.0 which is incompatible.
google-adk 2.7.1 requires starlette<2,>=1.3.1, but you have starlette 0.52.1 which is incompatible.
python-fasthtml 0.14.18 requires starlette>=1.0.1, but you have starlette 0.52.1 which is incompatible.
✅ Dependencies installed. If Colab asks to restart the runtime, restart it and continue from Cell 3.


In [2]:
import os, re, json, time, uuid, io, html, getpass
from datetime import datetime
from typing import TypedDict, List, Dict, Any, Optional

import numpy as np
import pandas as pd
import requests
import faiss
import pymupdf as fitz               # PyMuPDF (PDF reading)
import docx                          # python-docx (Word reading)
import openai
from openai import OpenAI
import plotly.graph_objects as go
import gradio as gr
from sentence_transformers import SentenceTransformer
from langgraph.graph import StateGraph, END

try:
    from ddgs import DDGS                      # new package name
except ImportError:
    from duckduckgo_search import DDGS         # older name

print("✅ Imports OK | Gradio", gr.__version__)

✅ Imports OK | Gradio 5.50.0


In [3]:
import os, getpass

# ================== EDIT THESE TWO LINES ==================
BASE_URL = "https://nexusapi.navigatelabs.ai"   # your provider's base URL (must start with https://)
MODEL    = "nova-micro"                          # your model ID (Cell 5 will list valid ones if this is wrong)
# ==========================================================

def load_api_key():
    """Order: Colab Secret (API_KEY or LLM_API_KEY) -> environment variable -> hidden prompt."""
    key = ""
    try:
        from google.colab import userdata
        for name in ("API_KEY", "LLM_API_KEY"):
            try:
                key = userdata.get(name)
                if key:
                    break
            except Exception:          # secret missing or notebook access not enabled
                continue
    except ImportError:
        pass
    key = key or os.environ.get("API_KEY", "") or os.environ.get("LLM_API_KEY", "")
    if not key:
        key = getpass.getpass("Secret not found. Paste your API key here (hidden): ")
    return (key or "").strip()

API_KEY = load_api_key()
if not API_KEY:
    raise ValueError("API key not found. Add a Colab Secret named API_KEY (🔑 icon, enable notebook access) or paste it when asked.")

CONFIG = {
    "api_key": API_KEY,
    "base_url": BASE_URL.strip(),
    "model": MODEL.strip(),
    "chunk_size": 800,
    "chunk_overlap": 150,
    "top_k": 5,
    "temperature": 0.2,
}

print("✅ API key loaded (hidden)")
print("Base URL:", CONFIG["base_url"])
print("Model   :", CONFIG["model"])

✅ API key loaded (hidden)
Base URL: https://nexusapi.navigatelabs.ai
Model   : nova-micro


In [4]:
class LLMError(Exception):
    """Friendly, user-facing LLM error."""

def normalize_base_url(url):
    """Trim spaces, add https:// if missing, remove trailing slash."""
    url = (url or "").strip().strip('"').strip("'")
    if url and not re.match(r"^https?://", url, re.I):
        url = "https://" + url
    return url.rstrip("/")

def key_status():
    k = CONFIG.get("api_key", "")
    return "🔑 API key: set (hidden)" if k else "🔑 API key: NOT set"

def validate_config():
    if not CONFIG.get("api_key", "").strip():
        raise LLMError("API key is missing. Open the Settings page and enter your API key.")
    CONFIG["base_url"] = normalize_base_url(CONFIG.get("base_url", ""))
    if not CONFIG["base_url"]:
        raise LLMError("Base URL is missing. Enter it in Settings (example: https://api.openai.com/v1).")
    if not CONFIG.get("model", "").strip():
        raise LLMError("Model name is missing. Enter a model name in Settings.")

def get_client():
    validate_config()
    return OpenAI(
        api_key=CONFIG["api_key"].strip(),
        base_url=CONFIG["base_url"],
        timeout=90,
        max_retries=1,
    )

def list_models():
    """Ask the provider which model IDs your key can use (helps fix 'model not found')."""
    try:
        data = get_client().models.list()
        return sorted(m.id for m in data.data)
    except Exception:
        return []

def explain_api_error(e):
    if isinstance(e, LLMError):
        msg = str(e)
    elif isinstance(e, openai.AuthenticationError):
        msg = "Invalid API key. Please check your key in Settings."
    elif isinstance(e, openai.NotFoundError):
        detail = getattr(e, "message", "") or str(e)
        msg = (f"Model '{CONFIG['model']}' or the endpoint was not found (server said: {detail[:200]}). "
               f"Check the Base URL (usually ends with /v1) and the model name in Settings.")
    elif isinstance(e, openai.RateLimitError):
        msg = "Rate limit or quota reached for this API key. Wait a little or check your provider plan."
    elif isinstance(e, openai.APITimeoutError):
        msg = "The API took too long to respond. Please try again."
    elif isinstance(e, openai.APIConnectionError):
        msg = "Could not connect to the API. Check the Base URL and your internet connection."
    elif isinstance(e, openai.BadRequestError):
        msg = f"The API rejected the request: {e}"
    else:
        msg = f"LLM error: {e}"
    key = CONFIG.get("api_key", "")
    if key:
        msg = msg.replace(key, "***")       # never leak the key
    return msg

def call_llm(messages, temperature=None, max_tokens=1500):
    """Send messages to the configured chat model and return the text answer."""
    client = get_client()
    temp = CONFIG["temperature"] if temperature is None else temperature
    model = CONFIG["model"].strip()
    try:
        try:
            r = client.chat.completions.create(model=model, messages=messages,
                                               temperature=temp, max_tokens=max_tokens)
        except openai.BadRequestError as e:
            # Some newer models need max_completion_tokens and no temperature
            if "max_completion_tokens" in str(e) or "temperature" in str(e):
                r = client.chat.completions.create(model=model, messages=messages,
                                                   max_completion_tokens=max_tokens)
            else:
                raise
    except Exception as e:
        raise LLMError(explain_api_error(e)) from None
    try:
        text = r.choices[0].message.content
    except Exception:
        raise LLMError("The model returned an unexpected response format.") from None
    if not text or not text.strip():
        raise LLMError("The model returned an empty answer. Try again or choose another model.")
    return text.strip()

def test_connection():
    try:
        out = call_llm([{"role": "user", "content": "Reply with the single word: OK"}],
                       temperature=0, max_tokens=50)
        return True, f"✅ Connected. Model '{CONFIG['model']}' replied: {out[:40]}"
    except LLMError as e:
        msg = f"❌ {e}"
        models = list_models()
        if models:
            msg += "\n\n📋 Models available for your key (copy one into MODEL in Cell 4 or into Settings):\n- " + "\n- ".join(models[:40])
        return False, msg

def extract_json(text):
    """Pull the first JSON object/array out of an LLM answer (handles ``` fences)."""
    t = re.sub(r"```(?:json)?", "", text).strip()
    starts = [i for i in (t.find("["), t.find("{")) if i != -1]
    if not starts:
        raise ValueError("No JSON found in model answer.")
    s = min(starts)
    e = max(t.rfind("]"), t.rfind("}"))
    return json.loads(t[s:e + 1])

if CONFIG["api_key"]:
    ok, info = test_connection()
    print(info)
    if not ok:
        ms = list_models()
        if ms and CONFIG["model"] not in ms:
            print("\n⚠️ Your MODEL name is not in the list above. Change MODEL in Cell 4 and re-run Cells 4 and 5.")
else:
    print("ℹ️ No API key yet. You can add it later in the Settings page.")

✅ Connected. Model 'nova-micro' replied: OK


In [5]:
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL_NAME)
EMBED_DIM = (embedder.get_embedding_dimension() if hasattr(embedder, "get_embedding_dimension")
             else embedder.get_sentence_embedding_dimension())

def embed_texts(texts, batch_size=32):
    """Return L2-normalised float32 vectors, so inner product = cosine similarity."""
    vecs = embedder.encode(list(texts), batch_size=batch_size, convert_to_numpy=True,
                           normalize_embeddings=True, show_progress_bar=False)
    return np.asarray(vecs, dtype="float32")

print("✅ Embedding model ready | dimension:", EMBED_DIM)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model ready | dimension: 384


In [6]:
def _search_ddgs(query, max_results):
    """Try several DDGS backends, because a single one is often rate-limited or blocked."""
    last = None
    for backend in ("auto", "duckduckgo", "bing", "brave", "mojeek", "yahoo"):
        try:
            with DDGS() as d:
                raw = list(d.text(query, max_results=max_results, backend=backend))
        except TypeError:                         # older package without the backend argument
            try:
                with DDGS() as d:
                    raw = list(d.text(query, max_results=max_results))
            except Exception as e:
                last = e; continue
        except Exception as e:
            last = e; continue
        out = []
        for r in raw:
            url = r.get("href") or r.get("url") or ""
            if url.startswith("http"):
                out.append({"title": (r.get("title") or url).strip(), "url": url,
                            "snippet": (r.get("body") or "").strip()})
        if out:
            return out
    return []

def _search_wikipedia(query, max_results):
    """Free fallback that works from Colab when DuckDuckGo blocks the request."""
    r = requests.get("https://en.wikipedia.org/w/api.php", timeout=10,
                     headers={"User-Agent": "KnowledgeAssistant/1.0 (Colab)"},
                     params={"action": "query", "list": "search", "srsearch": query,
                             "srlimit": max_results, "format": "json"})
    r.raise_for_status()
    out = []
    for it in r.json().get("query", {}).get("search", []):
        title = it["title"]
        snippet = re.sub(r"<[^>]+>", "", it.get("snippet", ""))
        out.append({"title": f"{title} - Wikipedia",
                    "url": "https://en.wikipedia.org/wiki/" + title.replace(" ", "_"),
                    "snippet": html.unescape(snippet)})
    return out

def web_search(query, max_results=6):
    errors = []
    try:
        res = _search_ddgs(query, max_results)
        if res:
            return res
        errors.append("DuckDuckGo returned no results")
    except Exception as e:
        errors.append(f"DuckDuckGo: {e}")
    try:
        res = _search_wikipedia(query, max_results)
        if res:
            return res
        errors.append("Wikipedia returned no results")
    except Exception as e:
        errors.append(f"Wikipedia: {e}")
    raise RuntimeError("Web search failed (" + "; ".join(errors) + "). Try a shorter query or press Retry.")

In [7]:
SUPPORTED_TYPES = {".pdf": "PDF", ".docx": "DOCX", ".txt": "TXT"}

class ExtractionError(Exception):
    pass

def clean_text(t):
    t = t.replace("\x00", " ")
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

def extract_pdf(path):
    """Returns (pages, total_pages). pages = [{'page': 1, 'text': '...'}, ...]"""
    pages, total = [], 0
    try:
        with fitz.open(path) as pdf:
            if pdf.needs_pass:
                raise ExtractionError("This PDF is password protected.")
            total = len(pdf)
            for i, page in enumerate(pdf, start=1):
                txt = clean_text(page.get_text("text") or "")
                if txt:
                    pages.append({"page": i, "text": txt})
    except ExtractionError:
        raise
    except Exception as e:
        raise ExtractionError(f"Could not read this PDF (it may be corrupted): {e}")
    if not pages:
        raise ExtractionError("No selectable text found. This may be a scanned PDF (OCR is not supported).")
    return pages, total

def extract_docx(path):
    try:
        d = docx.Document(path)
        parts = [p.text for p in d.paragraphs if p.text.strip()]
        for table in d.tables:
            for row in table.rows:
                cells = [c.text.strip() for c in row.cells if c.text.strip()]
                if cells:
                    parts.append(" | ".join(cells))
    except Exception as e:
        raise ExtractionError(f"Could not read this DOCX file (it may be corrupted): {e}")
    text = clean_text("\n".join(parts))
    if not text:
        raise ExtractionError("The DOCX file contains no readable text.")
    return [{"page": None, "text": text}], None      # DOCX has no fixed pages

def extract_txt(path):
    raw = None
    for enc in ("utf-8", "utf-16", "latin-1"):
        try:
            with open(path, "r", encoding=enc) as f:
                raw = f.read()
            break
        except Exception:
            continue
    if raw is None:
        raise ExtractionError("Could not read this TXT file.")
    text = clean_text(raw)
    if not text:
        raise ExtractionError("The TXT file is empty.")
    return [{"page": None, "text": text}], None

def extract_document(path):
    """Detects the file type and returns a dict with name, type, pages and page count."""
    name = os.path.basename(path)
    ext = os.path.splitext(name)[1].lower()
    if ext not in SUPPORTED_TYPES:
        raise ExtractionError(f"Unsupported file type '{ext or 'unknown'}'. Use PDF, DOCX or TXT.")
    if ext == ".pdf":
        pages, total = extract_pdf(path)
    elif ext == ".docx":
        pages, total = extract_docx(path)
    else:
        pages, total = extract_txt(path)
    return {"name": name, "type": SUPPORTED_TYPES[ext], "pages_data": pages, "num_pages": total}

print("✅ Extraction functions ready")

✅ Extraction functions ready


In [8]:
def chunk_text(text, chunk_size=800, overlap=150):
    """Split text into overlapping chunks, preferring paragraph/sentence boundaries."""
    chunk_size = max(200, int(chunk_size))
    overlap = max(0, min(int(overlap), chunk_size // 2))
    text = text.strip()
    if not text:
        return []
    if len(text) <= chunk_size:
        return [text]
    chunks, start, n = [], 0, len(text)
    while start < n:
        end = min(start + chunk_size, n)
        if end < n:
            window = text[start:end]
            cut = max(window.rfind("\n\n"), window.rfind(". "), window.rfind("\n"))
            if cut > chunk_size * 0.5:
                end = start + cut + 1
        piece = text[start:end].strip()
        if piece:
            chunks.append(piece)
        if end >= n:
            break
        start = max(end - overlap, start + 1)
    return chunks

def build_chunks(doc_name, pages, chunk_size=None, overlap=None):
    """Create chunk metadata: document name, page number (or None), chunk id, chunk text."""
    chunk_size = chunk_size or CONFIG["chunk_size"]
    overlap = CONFIG["chunk_overlap"] if overlap is None else overlap
    out, counter = [], 0
    for p in pages:
        for piece in chunk_text(p["text"], chunk_size, overlap):
            counter += 1
            out.append({"doc": doc_name, "page": p["page"],
                        "chunk_id": f"{doc_name}::chunk{counter}", "text": piece})
    return out

# quick self-check
_demo = build_chunks("demo.txt", [{"page": None, "text": "Hello world. " * 200}], 400, 80)
print("✅ Chunking works | demo chunks:", len(_demo), "| first chunk id:", _demo[0]["chunk_id"])

✅ Chunking works | demo chunks: 9 | first chunk id: demo.txt::chunk1


In [9]:
class VectorStore:
    """FAISS index + parallel list of chunk metadata."""
    def __init__(self, dim):
        self.dim = dim
        self.reset()

    def reset(self):
        self.index = faiss.IndexFlatIP(self.dim)   # inner product on normalised vectors = cosine
        self.chunks = []

    def add(self, chunks):
        if not chunks:
            return
        vecs = embed_texts([c["text"] for c in chunks])
        self.index.add(vecs)
        self.chunks.extend(chunks)

    def rebuild(self, chunks):
        """Throw away the old index and build a fresh one (used after add/remove/re-index)."""
        self.reset()
        self.add(chunks)

    def search(self, query, k=5, doc_names=None):
        if self.index.ntotal == 0:
            return []
        qv = embed_texts([query])
        n = self.index.ntotal if doc_names else min(k, self.index.ntotal)
        scores, ids = self.index.search(qv, n)
        hits = []
        for s, i in zip(scores[0], ids[0]):
            if i < 0:
                continue
            c = self.chunks[i]
            if doc_names and c["doc"] not in doc_names:
                continue
            hits.append({**c, "score": float(s)})
            if len(hits) >= k:
                break
        return hits

STORE = VectorStore(EMBED_DIM)

# Session document library: filename -> info dict (managed in Cell 14)
DOC_LIBRARY = {}

print("✅ FAISS vector store ready")

✅ FAISS vector store ready


In [10]:
def citation(hit):
    if hit.get("page"):
        return f"[Source: {hit['doc']}, Page {hit['page']}]"
    return f"[Source: {hit['doc']}, page unavailable]"

def retrieve(query, k=None, doc_names=None):
    """Embed the query and fetch the most similar chunks from FAISS."""
    k = k or CONFIG["top_k"]
    try:
        return STORE.search(query, k, doc_names)
    except Exception as e:
        raise RuntimeError(f"Document retrieval failed: {e}")

def format_context(hits, max_chars=9000):
    """Numbered context block with a citation tag for every chunk."""
    blocks, used = [], 0
    for i, h in enumerate(hits, 1):
        block = f"({i}) {citation(h)}\n{h['text']}"
        if used + len(block) > max_chars and blocks:
            break
        blocks.append(block)
        used += len(block)
    return "\n\n".join(blocks)

def format_sources(hits):
    """Markdown list of unique sources with short supporting snippets."""
    if not hits:
        return ""
    seen, lines = set(), ["**📚 Sources & supporting snippets**"]
    for h in hits:
        key = (h["doc"], h.get("page"), h["chunk_id"])
        if key in seen:
            continue
        seen.add(key)
        snippet = re.sub(r"\s+", " ", h["text"]).strip()[:180]
        lines.append(f"- `{citation(h)}`: “{snippet}…”")
    return "\n".join(lines)

def sample_doc_chunks(doc_name, max_chars=9000):
    """Evenly spread chunks across a document (for summaries, notes and quizzes without a topic)."""
    chunks = [c for c in STORE.chunks if c["doc"] == doc_name]
    if not chunks:
        return []
    total = sum(len(c["text"]) for c in chunks)
    if total <= max_chars:
        return chunks
    per = max(1, int(len(chunks) * max_chars / total))
    idx = sorted(set(np.linspace(0, len(chunks) - 1, per).astype(int).tolist()))
    return [chunks[i] for i in idx]

def get_doc_text(doc_name):
    """Full extracted text of a document (used by the comparison agent)."""
    info = DOC_LIBRARY.get(doc_name)
    if not info:
        raise RuntimeError(f"Document '{doc_name}' was not found in the library.")
    return "\n\n".join(p["text"] for p in info["pages_data"])

print("✅ RAG functions ready")

✅ RAG functions ready


In [11]:
PROMPTS = {
"coordinator": """You are the Main Coordinator of an AI knowledge assistant. Read the user's request and choose ONE agent.
Agents:
- knowledge: answer questions about the user's uploaded documents.
- study: explain topics, summarize, make notes, revision points, important questions, study plans or timetables.
- quiz: create MCQ quizzes / practice tests.
- research: search the internet / find current information online.
- comparison: compare two or more documents.
Rules: if documents exist and the question looks like it is about their content, choose knowledge. If no documents exist and it is a general concept question, choose study.
Reply ONLY with JSON: {"agent": "knowledge|study|quiz|research|comparison", "reason": "short reason"}""",

"knowledge": """You are the Knowledge Agent. Answer the question ONLY using the provided document context.
- Every factual statement must end with a citation tag copied exactly from the context, like [Source: file.pdf, Page 3].
- If the context does not contain the answer, reply exactly: "I could not find this in the uploaded documents." and optionally suggest a better question.
- Never invent facts, page numbers or sources. Use the conversation history only to understand follow-up questions.
- Be clear and well structured.""",

"study": """You are the Study Agent, a friendly tutor for college students.
Follow the requested TASK and STYLE exactly. Use headings, bullet points and tables where useful.
If document context is provided and relevant, base your answer on it and cite it like [Source: file.pdf, Page 3].
If no relevant context is provided, answer from general knowledge and say clearly that it is not from the uploaded documents.
Never invent page numbers or sources. Use conversation history to understand follow-ups ("explain that again", "give a short version").""",

"quiz": """You are the Quiz Agent. Create multiple-choice questions for students.
Return ONLY a JSON array (no markdown fences, no extra text). Each item must be:
{"question": "...", "options": ["option text", "option text", "option text", "option text"], "answer": "A", "explanation": "why it is correct"}
Rules: exactly 4 options per question; do NOT put letters like "A)" inside the option text; "answer" is one letter A, B, C or D;
the correct option must be spread across A-D; questions must be based on the given context when context is provided; distractors must be plausible; no duplicate questions.""",

"research": """You are the Research Agent. Write a research summary using ONLY the numbered web sources provided.
Format:
## Summary
(short paragraph)
## Key Findings
(bullet points; cite sources like [1], [2])
## Limitations
(one or two lines about what the sources do not cover)
Do not invent facts, sources or URLs. Do not write a source list; it is added automatically.""",

"comparison": """You are the Document Comparison Agent. Compare Document A and Document B using ONLY the text provided.
Format with these exact headings:
## Similarities
## Differences
## Unique Content in Document A
## Unique Content in Document B
## Topic Comparison
(a Markdown table with columns: Topic | Document A | Document B)
## Final Summary
Be specific. Never invent content that is not in the texts.""",

"segment_summary": """Summarize this part of a document in at most 250 words. Keep key topics, definitions, facts, names and numbers. Output only the summary.""",
}

STUDY_TASKS = {
    "Explain topic": "Explain the topic clearly and step by step.",
    "Generate notes": "Create well-organized study notes on the topic.",
    "Summarize document": "Write a clear summary of the document content.",
    "Unit-wise summary": "Create a unit-wise / section-wise summary of the document.",
    "Important topics & questions": "List the most important topics and likely exam questions with short hints.",
    "Revision points": "Create quick revision points: definitions, formulas, key facts, things to remember.",
    "Study plan / timetable": "Create a practical study timetable and exam preparation plan (day-wise, with revision slots).",
}

STUDY_STYLES = {
    "Simple English": "Use very simple English and short sentences, with an everyday example.",
    "Detailed": "Give a detailed, in-depth explanation with sub-headings and examples.",
    "Bullet points": "Use concise bullet points only.",
    "Exam preparation": "Format for exam preparation: definitions, key points, diagrams described in words, likely questions.",
    "Short notes": "Keep it very short: a 6 to 8 line note.",
    "10-mark answer": "Write a structured 10-mark exam answer (~350 words): introduction, main points with headings, conclusion.",
    "15-mark answer": "Write a structured 15-mark exam answer (~550 words): introduction, detailed headed sections, examples, conclusion.",
}

print("✅ Prompts ready:", ", ".join(PROMPTS.keys()))

✅ Prompts ready: coordinator, knowledge, study, quiz, research, comparison, segment_summary


In [12]:
# ---------- helpers ----------
def _mem(memory, n=6, limit=1200):
    """Last few chat turns, trimmed, as LLM messages."""
    out = []
    for m in (memory or [])[-n:]:
        if m.get("role") in ("user", "assistant") and m.get("content"):
            out.append({"role": m["role"], "content": str(m["content"])[:limit]})
    return out

def ask_llm(system, user, memory=None, max_tokens=1500, temperature=None):
    msgs = [{"role": "system", "content": system}] + _mem(memory) + [{"role": "user", "content": user}]
    return call_llm(msgs, temperature, max_tokens)

def _result(agent, answer, sources=""):
    return {"agent": agent, "answer": answer, "sources": sources}

def safe_run(label, fn, *args, **kwargs):
    """Run an agent; turn any failure into a friendly message instead of crashing the website."""
    try:
        return fn(*args, **kwargs)
    except LLMError as e:
        return _result(label, f"⚠️ {e}")
    except Exception as e:
        msg = str(e)
        if CONFIG["api_key"]:
            msg = msg.replace(CONFIG["api_key"], "***")
        return _result(label, f"⚠️ {label} could not finish this request: {msg}")

# ---------- Agent 2: Knowledge Agent ----------
def knowledge_agent(query, memory=None, doc_names=None):
    if not DOC_LIBRARY:
        return _result("Knowledge Agent", "📂 No documents are uploaded yet. Go to **Document Library**, upload a PDF, DOCX or TXT file, and ask again.")
    search_q = query
    if memory and len(query.split()) < 7:           # short follow-up: add previous question for better retrieval
        prev = [m["content"] for m in memory if m["role"] == "user"]
        if prev:
            search_q = prev[-1] + " " + query
    hits = retrieve(search_q, doc_names=doc_names)
    if not hits:
        return _result("Knowledge Agent", "I could not find this in the uploaded documents.")
    user = f"Document context:\n{format_context(hits)}\n\nQuestion: {query}"
    answer = ask_llm(PROMPTS["knowledge"], user, memory, max_tokens=1400)
    return _result("Knowledge Agent", answer, format_sources(hits))

# ---------- Agent 3: Study Agent ----------
def infer_study_task(q):
    ql = q.lower()
    if re.search(r"study plan|timetable|time table|schedule|exam preparation plan", ql): return "Study plan / timetable"
    if re.search(r"unit[- ]?wise|chapter[- ]?wise|section[- ]?wise", ql):                return "Unit-wise summary"
    if re.search(r"summar", ql):                                                         return "Summarize document"
    if re.search(r"important (topics|questions)|likely questions", ql):                  return "Important topics & questions"
    if re.search(r"revision|revise", ql):                                                return "Revision points"
    if re.search(r"\bnotes?\b", ql):                                                     return "Generate notes"
    return "Explain topic"

def study_agent(query, task="Explain topic", style="Simple English", doc_name=None, memory=None):
    task = task if task in STUDY_TASKS else "Explain topic"
    style = style if style in STUDY_STYLES else "Simple English"
    hits = []
    if doc_name and doc_name in DOC_LIBRARY:
        broad = task in ("Summarize document", "Unit-wise summary", "Study plan / timetable", "Revision points", "Important topics & questions")
        if query.strip() and not (broad and len(query.split()) < 3):
            hits = retrieve(query, k=8, doc_names=[doc_name])
        if not hits:
            hits = sample_doc_chunks(doc_name, 9000)
    elif DOC_LIBRARY and query.strip():
        hits = retrieve(query, k=6)
    context = format_context(hits) if hits else "None (no relevant uploaded-document context)."
    user = (f"TASK: {STUDY_TASKS[task]}\nSTYLE: {STUDY_STYLES[style]}\n"
            f"TOPIC / REQUEST: {query or 'the selected document'}\n\nDocument context:\n{context}")
    answer = ask_llm(PROMPTS["study"], user, memory, max_tokens=2200)
    return _result("Study Agent", answer, format_sources(hits))

# ---------- Agent 4: Quiz Agent ----------
def _clean_questions(data):
    if isinstance(data, dict):
        data = data.get("questions") or data.get("quiz") or []
    out = []
    for q in data:
        try:
            opts = [re.sub(r"^\s*[A-Da-d][\)\.\:]\s*", "", str(o)).strip() for o in q["options"]]
            ans = str(q["answer"]).strip().upper()[:1]
            if len(opts) != 4 or not ans or ans not in "ABCD":
                continue
            out.append({"question": str(q["question"]).strip(), "options": opts, "answer": ans,
                        "explanation": str(q.get("explanation", "")).strip()})
        except Exception:
            continue
    return out

def generate_quiz(doc_name=None, topic="", num_questions=5):
    """Returns a list of validated MCQ dicts. Answers are stored in the dicts, not shown to the user yet."""
    topic = (topic or "").strip()
    n = max(1, min(20, int(num_questions)))
    if doc_name and doc_name in DOC_LIBRARY:
        hits = retrieve(topic, k=10, doc_names=[doc_name]) if topic else sample_doc_chunks(doc_name, 8000)
    elif topic and DOC_LIBRARY:
        hits = retrieve(topic, k=8)
    elif topic:
        hits = []
    else:
        raise ValueError("Please select a document or enter a topic for the quiz.")
    context = format_context(hits, 8000) if hits else "None. Use reliable general knowledge about the topic."
    user = (f"Create exactly {n} multiple-choice questions."
            f"{' Topic: ' + topic + '.' if topic else ''}\n\nContext:\n{context}")
    questions = []
    for attempt in range(2):
        raw = call_llm([{"role": "system", "content": PROMPTS["quiz"]},
                        {"role": "user", "content": user if attempt == 0 else user + "\n\nReturn ONLY valid JSON, nothing else."}],
                       temperature=0.4, max_tokens=min(6000, 300 + 260 * n))
        try:
            questions = _clean_questions(extract_json(raw))
        except Exception:
            questions = []
        if questions:
            break
    if not questions:
        raise ValueError("Quiz generation failed because the model did not return valid questions. Please try again.")
    return questions[:n]

def format_quiz_text(questions):
    """Questions only (no answers), used in chat."""
    lines = []
    for i, q in enumerate(questions, 1):
        lines.append(f"**Q{i}. {q['question']}**")
        for letter, opt in zip("ABCD", q["options"]):
            lines.append(f"- {letter}. {opt}")
        lines.append("")
    return "\n".join(lines)

def grade_quiz(questions, answers):
    """answers = list of selected letters ('A'..'D') or None. Returns score, percentage, feedback and details."""
    score, blocks = 0, []
    for i, q in enumerate(questions):
        chosen = answers[i] if i < len(answers) else None
        ok = (chosen == q["answer"])
        score += int(ok)
        correct_text = q["options"]["ABCD".index(q["answer"])]
        chosen_text = q["options"]["ABCD".index(chosen)] if chosen in list("ABCD") else "Not answered"
        blocks.append(
            f"### {'✅' if ok else '❌'} Q{i+1}. {q['question']}\n"
            f"- **Your answer:** {chosen + '. ' + chosen_text if chosen else 'Not answered'}\n"
            f"- **Correct answer:** {q['answer']}. {correct_text}\n"
            f"- **Explanation:** {q['explanation'] or 'No explanation provided.'}\n")
    total = len(questions)
    pct = round(100 * score / total, 1) if total else 0.0
    if pct >= 80:   feedback = "🌟 Excellent! You have a strong understanding of this material."
    elif pct >= 60: feedback = "👍 Good job! Review the questions you missed to improve further."
    elif pct >= 40: feedback = "📘 Fair attempt. Revisit the topic notes and try another quiz."
    else:           feedback = "💪 Keep practicing! Re-read the material, then generate a new quiz."
    return {"score": score, "total": total, "percentage": pct, "feedback": feedback, "details": "\n".join(blocks)}

LAST_QUIZ = {"questions": [], "title": ""}     # the Quiz Center can load a quiz made in chat

def quiz_agent(query, memory=None, doc_names=None):
    m = re.search(r"\b(\d{1,2})\s*(?:questions?|mcqs?|qs?)\b", query, re.I)
    n = max(1, min(20, int(m.group(1)))) if m else 5
    topic = re.sub(r"(?i)\b(create|generate|make|give|me|a|an|the|quiz|mcqs?|multiple|choice|questions?|test|about|on|from|of|please|\d+)\b", " ", query)
    topic = re.sub(r"\s+", " ", topic).strip()
    if memory and re.search(r"\b(this|that|above|previous|it)\b", query, re.I):
        prev = [x["content"] for x in memory if x["role"] == "assistant"]
        if prev:
            topic = prev[-1][:1500]
    doc = doc_names[0] if doc_names else None
    if not doc and not topic and DOC_LIBRARY:
        doc = list(DOC_LIBRARY)[-1]                  # newest document
    questions = generate_quiz(doc, topic, n)
    LAST_QUIZ["questions"] = questions
    LAST_QUIZ["title"] = f"{doc or topic[:40]} ({len(questions)} questions)"
    text = (f"📝 I created **{len(questions)} MCQs**. Open the **Quiz Center** to answer them interactively and get your score.\n\n"
            + format_quiz_text(questions))
    return _result("Quiz Agent", text)

# ---------- Agent 5: Research Agent (FIXED web search) ----------
def _search_ddgs(query, max_results):
    """Try several DDGS backends, because a single one is often rate-limited or blocked."""
    last = None
    for backend in ("auto", "duckduckgo", "bing", "brave", "mojeek", "yahoo"):
        try:
            with DDGS() as d:
                raw = list(d.text(query, max_results=max_results, backend=backend))
        except TypeError:                         # older package without the backend argument
            try:
                with DDGS() as d:
                    raw = list(d.text(query, max_results=max_results))
            except Exception as e:
                last = e; continue
        except Exception as e:
            last = e; continue
        out = []
        for r in raw:
            url = r.get("href") or r.get("url") or ""
            if url.startswith("http"):
                out.append({"title": (r.get("title") or url).strip(), "url": url,
                            "snippet": (r.get("body") or "").strip()})
        if out:
            return out
    return []

def _search_wikipedia(query, max_results):
    """Free fallback that works from Colab when DuckDuckGo blocks the request."""
    r = requests.get("https://en.wikipedia.org/w/api.php", timeout=10,
                     headers={"User-Agent": "KnowledgeAssistant/1.0 (Colab)"},
                     params={"action": "query", "list": "search", "srsearch": query,
                             "srlimit": max_results, "format": "json"})
    r.raise_for_status()
    out = []
    for it in r.json().get("query", {}).get("search", []):
        title = it["title"]
        snippet = re.sub(r"<[^>]+>", "", it.get("snippet", ""))
        out.append({"title": f"{title} - Wikipedia",
                    "url": "https://en.wikipedia.org/wiki/" + title.replace(" ", "_"),
                    "snippet": html.unescape(snippet)})
    return out

def web_search(query, max_results=6):
    errors = []
    try:
        res = _search_ddgs(query, max_results)
        if res:
            return res
        errors.append("DuckDuckGo returned no results")
    except Exception as e:
        errors.append(f"DuckDuckGo: {e}")
    try:
        res = _search_wikipedia(query, max_results)
        if res:
            return res
        errors.append("Wikipedia returned no results")
    except Exception as e:
        errors.append(f"Wikipedia: {e}")
    raise RuntimeError("Web search failed (" + "; ".join(errors) + "). Try a shorter query or press Retry.")

def fetch_page_text(url, max_chars=2500):
    """Try to read page text for a richer summary; silently skip pages that fail."""
    try:
        r = requests.get(url, timeout=8, headers={"User-Agent": "Mozilla/5.0"})
        if r.status_code != 200 or "text/html" not in r.headers.get("content-type", ""):
            return ""
        t = re.sub(r"(?is)<(script|style|noscript|header|footer|nav).*?>.*?</\1>", " ", r.text)
        t = re.sub(r"(?s)<[^>]+>", " ", t)
        return re.sub(r"\s+", " ", html.unescape(t)).strip()[:max_chars]
    except Exception:
        return ""

def research_agent(query, memory=None):
    results = web_search(query)
    blocks = []
    for i, r in enumerate(results, 1):
        extra = fetch_page_text(r["url"]) if i <= 3 else ""
        blocks.append(f"[{i}] {r['title']}\nURL: {r['url']}\nSnippet: {r['snippet']}\n{('Page text: ' + extra) if extra else ''}")
    user = f"Research topic: {query}\n\nWeb sources:\n\n" + "\n\n".join(blocks)
    summary = ask_llm(PROMPTS["research"], user, memory, max_tokens=1600)
    src = "\n".join(f"{i}. [{r['title']}]({r['url']})" for i, r in enumerate(results, 1))   # links come from the real search, not the LLM
    return _result("Research Agent", summary,
                   f"**🌐 Online sources** *(from the web, not from your uploaded documents)*\n\n{src}")

# ---------- Agent 6: Document Comparison Agent ----------
def condense_document(name, target_chars=7000, seg_size=8000, max_segments=6):
    """Long documents are summarised in segments (map step) so both texts fit in the LLM context."""
    text = get_doc_text(name)
    if len(text) <= target_chars:
        return text
    segs = [text[i:i + seg_size] for i in range(0, len(text), seg_size)]
    if len(segs) > max_segments:
        idx = sorted(set(np.linspace(0, len(segs) - 1, max_segments).astype(int).tolist()))
        segs = [segs[i] for i in idx]
    parts = [call_llm([{"role": "system", "content": PROMPTS["segment_summary"]},
                       {"role": "user", "content": s}], temperature=0.2, max_tokens=600) for s in segs]
    return "\n\n".join(parts)

def comparison_agent(doc_a, doc_b):
    if doc_a not in DOC_LIBRARY or doc_b not in DOC_LIBRARY:
        return _result("Document Comparison Agent", "⚠️ Please select two documents that exist in your Document Library.")
    if doc_a == doc_b:
        return _result("Document Comparison Agent", "⚠️ Please choose two **different** documents.")
    ta, tb = condense_document(doc_a), condense_document(doc_b)
    user = f"DOCUMENT A: {doc_a}\n{ta}\n\n-----\n\nDOCUMENT B: {doc_b}\n{tb}"
    answer = ask_llm(PROMPTS["comparison"], user, None, max_tokens=2600)
    return _result("Document Comparison Agent", answer, f"**📄 Compared:** `{doc_a}` vs `{doc_b}`")

print("✅ All agent functions ready")

✅ All agent functions ready


In [13]:
AGENT_LABELS = {
    "knowledge": "Knowledge Agent",
    "study": "Study Agent",
    "quiz": "Quiz Agent",
    "research": "Research Agent",
    "comparison": "Document Comparison Agent",
}
LABEL_TO_KEY = {v: k for k, v in AGENT_LABELS.items()}
AGENT_CHOICES = ["Auto Agent"] + list(AGENT_LABELS.values())

class CoordinatorState(TypedDict, total=False):
    query: str
    memory: list
    forced_agent: str
    doc_names: list
    study_task: str
    study_style: str
    route: str
    reason: str
    result: dict

# ---------- routing logic ----------
def heuristic_route(q, has_docs):
    ql = q.lower()
    if re.search(r"\bcompar(e|ison|ing)\b|similarities and differences|differences? between (the )?(two )?(documents?|files?)", ql):
        return "comparison", "Keyword rule: comparison request"
    if re.search(r"\b(quiz|mcqs?|multiple[- ]choice|test me|practice questions)\b", ql):
        return "quiz", "Keyword rule: quiz request"
    if re.search(r"search (the )?(web|internet|online)|\bresearch\b|latest news|look up online|on the internet|\bgoogle\b|\bonline\b", ql):
        return "research", "Keyword rule: web research request"
    if re.search(r"\b(explain|summari[sz]e|notes?|study plan|timetable|revision|revise|exam|simple (words|english)|important questions|10[- ]mark|15[- ]mark|teach me|short version|shorter|simplify|rephrase)\b", ql):
        return "study", "Keyword rule: study request"
    return ("knowledge", "Question about uploaded documents") if has_docs else ("study", "General question (no documents uploaded)")

def llm_route(q, memory, has_docs):
    recent = " | ".join(f"{m['role']}: {str(m['content'])[:120]}" for m in (memory or [])[-4:])
    user = f"Documents uploaded: {'yes' if has_docs else 'no'}\nRecent conversation: {recent or 'none'}\nUser request: {q}"
    out = call_llm([{"role": "system", "content": PROMPTS["coordinator"]},
                    {"role": "user", "content": user}], temperature=0, max_tokens=150)
    data = extract_json(out)
    agent = str(data.get("agent", "")).lower().strip()
    if agent not in AGENT_LABELS:
        raise ValueError("Invalid agent from router")
    return agent, str(data.get("reason", "LLM routing"))

# ---------- graph nodes ----------
def coordinator_node(state):
    q = (state.get("query") or "").strip()
    forced = state.get("forced_agent") or "Auto Agent"
    if not q:
        return {"route": "unsupported", "reason": "Empty request"}
    if forced in LABEL_TO_KEY:
        return {"route": LABEL_TO_KEY[forced], "reason": "Agent selected by the user"}
    has_docs = bool(DOC_LIBRARY)

    # 1) Clear keyword requests (quiz / compare / web research) are routed by rule,
    #    so they are never misrouted to the Knowledge Agent
    rule_route, rule_reason = heuristic_route(q, has_docs)
    if rule_route in ("quiz", "comparison", "research"):
        return {"route": rule_route, "reason": rule_reason}

    # 2) Everything else: let the LLM decide, with keyword rules as the backup
    try:
        route, reason = llm_route(q, state.get("memory"), has_docs)
    except Exception:
        route, reason = rule_route, rule_reason
    return {"route": route, "reason": reason}

def route_decider(state):
    return state.get("route", "unsupported")

def knowledge_node(state):
    return {"result": safe_run("Knowledge Agent", knowledge_agent, state["query"], state.get("memory"), state.get("doc_names"))}

def study_node(state):
    q = state["query"]
    task = state.get("study_task") or infer_study_task(q)
    style = state.get("study_style") or "Simple English"
    doc = state["doc_names"][0] if state.get("doc_names") else None
    return {"result": safe_run("Study Agent", study_agent, q, task, style, doc, state.get("memory"))}

def quiz_node(state):
    return {"result": safe_run("Quiz Agent", quiz_agent, state["query"], state.get("memory"), state.get("doc_names"))}

def research_node(state):
    return {"result": safe_run("Research Agent", research_agent, state["query"], state.get("memory"))}

def pick_comparison_docs(query, doc_names):
    names = list(DOC_LIBRARY)
    ql = query.lower()
    found = [n for n in names if n.lower() in ql or os.path.splitext(n)[0].lower() in ql]
    if len(found) >= 2:
        return found[:2]
    if doc_names and len(doc_names) >= 2:
        return doc_names[:2]
    if len(names) == 2:
        return names
    return None

def comparison_node(state):
    names = list(DOC_LIBRARY)
    if len(names) < 2:
        return {"result": _result("Document Comparison Agent", "📂 Please upload **at least two documents** in the Document Library to compare them.")}
    pair = pick_comparison_docs(state["query"], state.get("doc_names"))
    if not pair:
        return {"result": _result("Document Comparison Agent",
                "You have several documents. Please mention two filenames in your message, or use the **Document Comparison** page to choose Document A and Document B.")}
    return {"result": safe_run("Document Comparison Agent", comparison_agent, pair[0], pair[1])}

def unsupported_node(state):
    return {"result": _result("Main Coordinator", "⚠️ Please type a question or request. Example: *“Summarize my uploaded notes”* or *“Create a 5-question quiz on Chapter 2”*.")}

def finalize_node(state):
    res = state.get("result") or _result("Main Coordinator", "⚠️ No agent produced a response. Please try again.")
    return {"result": res}

# ---------- build the LangGraph workflow ----------
def build_coordinator_graph():
    g = StateGraph(CoordinatorState)
    g.add_node("coordinator", coordinator_node)
    g.add_node("knowledge", knowledge_node)
    g.add_node("study", study_node)
    g.add_node("quiz", quiz_node)
    g.add_node("research", research_node)
    g.add_node("comparison", comparison_node)
    g.add_node("unsupported", unsupported_node)
    g.add_node("finalize", finalize_node)

    g.set_entry_point("coordinator")
    g.add_conditional_edges("coordinator", route_decider, {
        "knowledge": "knowledge", "study": "study", "quiz": "quiz",
        "research": "research", "comparison": "comparison", "unsupported": "unsupported",
    })
    for node in ("knowledge", "study", "quiz", "research", "comparison", "unsupported"):
        g.add_edge(node, "finalize")
    g.add_edge("finalize", END)
    return g.compile()

COORDINATOR = build_coordinator_graph()

def run_coordinator(query, memory=None, forced_agent="Auto Agent", doc_names=None,
                    study_task=None, study_style=None):
    """Main entry point used by the website. Never raises, always returns a result dict."""
    state = {"query": query or "", "memory": memory or [], "forced_agent": forced_agent,
             "doc_names": doc_names or [], "study_task": study_task, "study_style": study_style}
    try:
        out = COORDINATOR.invoke(state)
        res = dict(out.get("result") or {})
        res["route"] = out.get("route", "")
        res["reason"] = out.get("reason", "")
        return res
    except Exception as e:
        return {"agent": "Main Coordinator", "answer": f"⚠️ The coordinator hit an unexpected problem: {e}",
                "sources": "", "route": "", "reason": ""}

print("✅ LangGraph coordinator compiled | agents:", ", ".join(AGENT_LABELS.values()))

✅ LangGraph coordinator compiled | agents: Knowledge Agent, Study Agent, Quiz Agent, Research Agent, Document Comparison Agent


In [14]:
NO_DOC = "No document (topic only)"
ALL_DOCS = "All documents"
LIB_COLUMNS = ["Filename", "Format", "Status", "Pages", "Chunks", "Indexed", "Added"]

def _norm_paths(files):
    if not files:
        return []
    if not isinstance(files, (list, tuple)):
        files = [files]
    return [f if isinstance(f, str) else getattr(f, "name", str(f)) for f in files]

def rebuild_index():
    """Rebuild the FAISS index from every chunk of every document in the library."""
    all_chunks = []
    for info in DOC_LIBRARY.values():
        all_chunks.extend(info["chunks"])
    try:
        STORE.rebuild(all_chunks)
        for info in DOC_LIBRARY.values():
            info["status"] = "Indexed ✅"
    except Exception as e:
        for info in DOC_LIBRARY.values():
            info["status"] = "Index error ❌"
        raise RuntimeError(f"FAISS indexing failed: {e}")

def add_documents(files):
    paths = _norm_paths(files)
    if not paths:
        return "⚠️ Please choose at least one PDF, DOCX or TXT file first."
    good, bad = [], []
    for path in paths:
        name = os.path.basename(path)
        try:
            doc = extract_document(path)
            chunks = build_chunks(doc["name"], doc["pages_data"])
            if not chunks:
                raise ExtractionError("No text chunks could be created from this file.")
            DOC_LIBRARY[doc["name"]] = {**doc, "chunks": chunks, "status": "Processing",
                                        "added": datetime.now().strftime("%H:%M:%S")}
            good.append(f"✅ **{doc['name']}**: {doc['type']}, {len(chunks)} chunks")
        except ExtractionError as e:
            bad.append(f"❌ **{name}**: {e}")
        except Exception as e:
            bad.append(f"❌ **{name}**: unexpected error: {e}")
    try:
        rebuild_index()
    except Exception as e:
        bad.append(f"❌ {e}")
    return "\n\n".join(good + bad) or "⚠️ Nothing was processed."

def remove_document(name):
    if not name or name not in DOC_LIBRARY:
        return "⚠️ Select a document to remove."
    del DOC_LIBRARY[name]
    try:
        rebuild_index()
    except Exception as e:
        return f"❌ {e}"
    return f"🗑️ Removed **{name}** and rebuilt the search index."

def reindex_document(name):
    info = DOC_LIBRARY.get(name or "")
    if not info:
        return "⚠️ Select a document to re-index."
    try:
        info["chunks"] = build_chunks(name, info["pages_data"])
        rebuild_index()
        return f"🔄 Re-indexed **{name}**: {len(info['chunks'])} chunks."
    except Exception as e:
        return f"❌ Re-index failed: {e}"

def reindex_all():
    if not DOC_LIBRARY:
        return "ℹ️ The library is empty."
    try:
        for n, info in DOC_LIBRARY.items():
            info["chunks"] = build_chunks(n, info["pages_data"])
        rebuild_index()
        return f"✅ Re-indexed {len(DOC_LIBRARY)} document(s) with {len(STORE.chunks)} chunks."
    except Exception as e:
        return f"❌ Re-index failed: {e}"

def clear_library():
    DOC_LIBRARY.clear()
    STORE.reset()
    return "🧹 Document library cleared."

def view_document_text(name, limit=6000):
    info = DOC_LIBRARY.get(name or "")
    if not info:
        return "Select a document first."
    parts = []
    for p in info["pages_data"]:
        parts.append((f"--- Page {p['page']} ---\n" if p["page"] else "") + p["text"])
    text = "\n\n".join(parts)
    return text[:limit] + ("\n\n… (truncated)" if len(text) > limit else "")

def library_dataframe():
    rows = []
    for n, i in DOC_LIBRARY.items():
        rows.append([n, i["type"], i["status"], str(i["num_pages"]) if i["num_pages"] else "N/A",
                     len(i["chunks"]), "Yes" if "Indexed" in i["status"] else "No", i["added"]])
    return pd.DataFrame(rows, columns=LIB_COLUMNS)

def doc_choices():
    return list(DOC_LIBRARY.keys())

print("✅ Document library functions ready")

✅ Document library functions ready


In [15]:
MEMORY = []                       # raw {"role","content"} turns sent to the agents
HISTORY = []                      # log shown on the Chat History page
STATS = {"questions": 0, "quizzes": 0}
MEMORY_LIMIT = 20

def remember(role, content):
    MEMORY.append({"role": role, "content": str(content)})
    del MEMORY[:-MEMORY_LIMIT]

def recent_memory(n=8):
    return MEMORY[-n:]

def clear_memory():
    MEMORY.clear()

def log_history(kind, agent, title, content):
    HISTORY.append({"time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "kind": kind,
                    "agent": agent, "title": str(title)[:120], "content": str(content)})

def clear_history():
    HISTORY.clear()

def history_markdown():
    banner = ("> ℹ️ History and chat memory are stored **only in this Colab session**. "
              "They are **not permanent** and reset when the Colab runtime disconnects.")
    if not HISTORY:
        return banner + "\n\nNo history yet. Ask a question, generate notes, take a quiz or run research."
    parts = [banner]
    for h in reversed(HISTORY):
        body = h["content"][:3000]
        parts.append(f"### {h['time']} · {h['kind'].title()} · {h['agent']}\n**{h['title']}**\n\n"
                     f"<details><summary>Show response</summary>\n\n{body}\n\n</details>\n\n---")
    return "\n\n".join(parts)

print("✅ Chat memory ready")

✅ Chat memory ready


In [16]:
APP_TITLE = "AI Personal Knowledge Assistant project"

UI = {}
def reset_ui():
    UI.clear()
    UI.update({"pages": {}, "dash_events": [], "lib_events": []})
reset_ui()

def dashboard_stats():
    return {"docs": len(DOC_LIBRARY), "chunks": len(STORE.chunks),
            "questions": STATS["questions"], "quizzes": STATS["quizzes"], "agents": 6}

def dashboard_html():
    s = dashboard_stats()
    cards = [("📄", "Documents Uploaded", s["docs"]), ("🧩", "Chunks Indexed", s["chunks"]),
             ("❓", "Questions Asked", s["questions"]), ("📝", "Quizzes Generated", s["quizzes"]),
             ("🤖", "AI Agents", s["agents"])]
    card_html = "".join(
        f'<div class="stat-card"><div class="stat-icon">{i}</div>'
        f'<div class="stat-value">{v}</div><div class="stat-label">{l}</div></div>' for i, l, v in cards)
    chips = "".join(f'<span class="chip">{html.escape(n)}</span>'
                    for n in ["Main Coordinator"] + list(AGENT_LABELS.values()))
    return (f'<div class="hero"><div class="hero-logo">🧠</div><div>'
            f'<h1>{html.escape(APP_TITLE)}</h1>'
            f'<p>Welcome! Upload your notes, textbooks and papers, then ask questions, generate study notes, '
            f'take quizzes, research the web and compare documents. A LangGraph Main Coordinator routes every request '
            f'to the right specialised agent using Retrieval-Augmented Generation (FAISS + Sentence Transformers).</p>'
            f'<div class="chips">{chips}</div></div></div>'
            f'<div class="stat-grid">{card_html}</div>')

def activity_markdown():
    if not HISTORY:
        return "No activity yet. Your recent questions, notes, quizzes and research will appear here."
    lines = [f"- `{h['time'][11:]}` **{h['agent']}**: {h['title']}" for h in reversed(HISTORY[-6:])]
    return "\n".join(lines)

def chunks_figure():
    fig = go.Figure()
    if DOC_LIBRARY:
        names = list(DOC_LIBRARY)
        fig.add_bar(x=names, y=[len(DOC_LIBRARY[n]["chunks"]) for n in names],
                    marker=dict(color="#7c3aed"))
    else:
        fig.add_annotation(text="Upload documents to see statistics", showarrow=False, font=dict(size=16))
        fig.update_xaxes(visible=False)
        fig.update_yaxes(visible=False)
    fig.update_layout(template="plotly_dark", title="Indexed chunks per document",
                      paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                      height=320, margin=dict(l=20, r=20, t=50, b=20))
    return fig

def refresh_dashboard():
    return dashboard_html(), chunks_figure(), activity_markdown()

def build_dashboard_page():
    with gr.Column(visible=True, elem_classes="page") as col:
        UI["pages"]["Dashboard"] = col
        UI["dash_html"] = gr.HTML(dashboard_html())
        gr.Markdown("### ⚡ Quick actions")
        with gr.Row():
            UI["qa_chat"] = gr.Button("💬 Ask AI", elem_classes="qa-btn")
            UI["qa_upload"] = gr.Button("📤 Upload Documents", elem_classes="qa-btn")
            UI["qa_study"] = gr.Button("🎓 Generate Study Notes", elem_classes="qa-btn")
        with gr.Row():
            UI["qa_quiz"] = gr.Button("📝 Create Quiz", elem_classes="qa-btn")
            UI["qa_research"] = gr.Button("🌐 Research Topic", elem_classes="qa-btn")
            UI["qa_compare"] = gr.Button("⚖️ Compare Documents", elem_classes="qa-btn")
        with gr.Row():
            with gr.Column(scale=3):
                UI["dash_plot"] = gr.Plot(chunks_figure(), label="Library statistics")
            with gr.Column(scale=2):
                gr.Markdown("### 🕒 Recent activity")
                UI["dash_activity"] = gr.Markdown(activity_markdown())

print("✅ Dashboard ready")

✅ Dashboard ready


In [17]:
def chat_respond(message, history, agent_choice, doc_choice):
    history = list(history or [])
    message = (message or "").strip()
    if not message:
        yield history, "", "⚠️ Please type a message first."
        return
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": "⏳ **Thinking…** the Main Coordinator is choosing the best agent."})
    yield history, "", "🔄 Routing your request…"

    doc_names = [doc_choice] if doc_choice in DOC_LIBRARY else None
    res = run_coordinator(message, recent_memory(), agent_choice or "Auto Agent", doc_names)

    body = f"**🤖 {res['agent']}**\n\n{res['answer']}"
    if res.get("sources"):
        body += f"\n\n---\n{res['sources']}"
    history[-1] = {"role": "assistant", "content": body}

    remember("user", message)
    remember("assistant", res["answer"])
    STATS["questions"] += 1
    if res["agent"] == "Quiz Agent" and str(res["answer"]).startswith("📝"):
        STATS["quizzes"] += 1
    log_history("chat", res["agent"], message, res["answer"] + "\n\n" + res.get("sources", ""))

    note = f"✅ Handled by **{res['agent']}**"
    if res.get("reason"):
        note += f" · {res['reason']}"
    yield history, "", note

def chat_clear():
    clear_memory()
    return [], "🧹 Chat and conversation memory cleared."

def build_chat_page():
    with gr.Column(visible=False, elem_classes="page") as col:
        UI["pages"]["AI Knowledge Chat"] = col
        gr.Markdown("## 💬 AI Knowledge Chat\nAsk anything. In **Auto Agent** mode the Main Coordinator picks the best agent. "
                    "Follow-up questions such as *“Explain that again”* or *“Give a short version”* work during this session.")
        with gr.Row():
            UI["chat_agent"] = gr.Dropdown(AGENT_CHOICES, value="Auto Agent", label="Agent", scale=1)
            UI["chat_doc"] = gr.Dropdown([ALL_DOCS], value=ALL_DOCS, label="Limit to document", scale=1)
        UI["chatbot"] = gr.Chatbot(type="messages", height=500, show_copy_button=True,
                                   label="Conversation", elem_classes="chat-box")
        UI["chat_indicator"] = gr.Markdown("Agent indicator: waiting for your first message.")
        with gr.Row():
            UI["chat_input"] = gr.Textbox(show_label=False, placeholder="Type your question and press Enter…",
                                          lines=1, scale=6)
            UI["chat_send"] = gr.Button("Send ➤", variant="primary", scale=1)
            UI["chat_clear"] = gr.Button("🧹 Clear chat", scale=1)
        gr.Examples(examples=[["Summarize my uploaded documents"], ["Explain the main topic in simple words"],
                              ["Create a 5-question quiz from my document"], ["Search the web for the latest AI news"]],
                    inputs=UI["chat_input"], label="Try these")

        ins = [UI["chat_input"], UI["chatbot"], UI["chat_agent"], UI["chat_doc"]]
        outs = [UI["chatbot"], UI["chat_input"], UI["chat_indicator"]]
        e1 = UI["chat_send"].click(chat_respond, ins, outs)
        e2 = UI["chat_input"].submit(chat_respond, ins, outs)
        UI["chat_clear"].click(chat_clear, None, [UI["chatbot"], UI["chat_indicator"]])
        UI["dash_events"].extend([e1, e2])

print("✅ Chat interface ready")

✅ Chat interface ready


In [18]:
def study_generate(topic, doc, task, style):
    topic = (topic or "").strip()
    doc_name = doc if doc in DOC_LIBRARY else None
    if not topic and not doc_name:
        return "⚠️ Please enter a topic or select an uploaded document.", ""
    broad = task in ("Summarize document", "Unit-wise summary", "Study plan / timetable",
                     "Revision points", "Important topics & questions")
    if not topic and not broad:
        return "⚠️ Please enter a topic for this task (or choose a summary/revision task for the whole document).", ""
    query = topic or "the document"
    res = run_coordinator(query, None, "Study Agent", [doc_name] if doc_name else None, task, style)
    log_history("study", res["agent"], f"{task}: {topic or doc_name}", res["answer"] + "\n\n" + res.get("sources", ""))
    return res["answer"], res.get("sources", "")

def build_study_page():
    with gr.Column(visible=False, elem_classes="page") as col:
        UI["pages"]["Study Assistant"] = col
        gr.Markdown("## 🎓 Study Assistant\nGenerate notes, summaries, important questions, revision points or a study plan.")
        with gr.Row():
            UI["study_topic"] = gr.Textbox(label="Topic", placeholder="e.g. Overfitting in machine learning", scale=2)
            UI["study_doc"] = gr.Dropdown([NO_DOC], value=NO_DOC, label="Document", scale=1)
        with gr.Row():
            UI["study_task"] = gr.Dropdown(list(STUDY_TASKS), value="Explain topic", label="What do you need?")
            UI["study_style"] = gr.Dropdown(list(STUDY_STYLES), value="Simple English", label="Explanation style")
        UI["study_btn"] = gr.Button("✨ Generate", variant="primary")
        UI["study_out"] = gr.Markdown("Your study material will appear here.", elem_classes="output-card")
        UI["study_src"] = gr.Markdown()
        ev = UI["study_btn"].click(study_generate,
                                   [UI["study_topic"], UI["study_doc"], UI["study_task"], UI["study_style"]],
                                   [UI["study_out"], UI["study_src"]])
        UI["dash_events"].append(ev)

print("✅ Study interface ready")

✅ Study interface ready


In [19]:
MAX_Q = 20

def _quiz_hidden(msg):
    return [[], msg, gr.update(visible=False), gr.update(visible=False), ""] + \
           [gr.update(visible=False, value=None, interactive=True) for _ in range(MAX_Q)]

def _quiz_show(questions, note):
    radios = []
    for i in range(MAX_Q):
        if i < len(questions):
            q = questions[i]
            radios.append(gr.update(visible=True, label=f"Q{i+1}. {q['question']}",
                                    choices=[(f"{L}. {o}", L) for L, o in zip("ABCD", q["options"])],
                                    value=None, interactive=True))
        else:
            radios.append(gr.update(visible=False, value=None))
    return [questions, note, gr.update(visible=True), gr.update(visible=True), ""] + radios

def quiz_generate(doc, topic, n):
    try:
        doc_name = doc if doc in DOC_LIBRARY else None
        questions = generate_quiz(doc_name, topic, int(n))
    except (LLMError, ValueError, RuntimeError) as e:
        return _quiz_hidden(f"⚠️ {e}")
    except Exception as e:
        return _quiz_hidden(f"⚠️ Quiz generation failed: {e}")
    STATS["quizzes"] += 1
    return _quiz_show(questions, f"✅ Quiz ready: **{len(questions)} questions**. Choose one answer for each, then press **Submit quiz**.")

def quiz_load_from_chat():
    qs = LAST_QUIZ.get("questions") or []
    if not qs:
        return _quiz_hidden("ℹ️ No quiz from chat yet. Ask the chat: “Create a 5-question quiz on …”.")
    return _quiz_show(qs, f"✅ Loaded quiz from chat: {LAST_QUIZ.get('title', '')}")

def quiz_submit(questions, *answers):
    if not questions:
        return ["⚠️ Generate a quiz first."] + [gr.update() for _ in range(MAX_Q)]
    res = grade_quiz(questions, list(answers))
    unanswered = sum(1 for a in answers[:len(questions)] if a is None)
    summary = (f"## 🎯 Score: {res['score']} / {res['total']}  ({res['percentage']}%)\n\n{res['feedback']}\n\n"
               + (f"⚠️ You left {unanswered} question(s) unanswered.\n\n" if unanswered else "")
               + "---\n\n" + res["details"])
    log_history("quiz", "Quiz Agent", f"Quiz result {res['score']}/{res['total']}", summary)
    return [summary] + [gr.update(interactive=False) for _ in range(MAX_Q)]

def build_quiz_page():
    with gr.Column(visible=False, elem_classes="page") as col:
        UI["pages"]["Quiz Center"] = col
        gr.Markdown("## 📝 Quiz Center\nCreate MCQs from a document or a topic. Answers stay hidden until you submit.")
        with gr.Row():
            UI["quiz_doc"] = gr.Dropdown([NO_DOC], value=NO_DOC, label="Document", scale=2)
            UI["quiz_topic"] = gr.Textbox(label="Topic (optional)", placeholder="e.g. Neural networks", scale=2)
            UI["quiz_num"] = gr.Dropdown([5, 10, 15, 20], value=5, label="Questions", scale=1)
        with gr.Row():
            UI["quiz_gen"] = gr.Button("🎲 Generate quiz", variant="primary")
            UI["quiz_load"] = gr.Button("📥 Load quiz from chat")
        UI["quiz_status"] = gr.Markdown("Choose a document or topic and press **Generate quiz**.")
        UI["quiz_state"] = gr.State([])
        UI["quiz_radios"] = [gr.Radio(choices=[], label="", visible=False, elem_classes="quiz-q") for _ in range(MAX_Q)]
        with gr.Row():
            UI["quiz_submit"] = gr.Button("✅ Submit quiz", variant="primary", visible=False)
            UI["quiz_again"] = gr.Button("🔁 Generate another quiz", visible=False)
        UI["quiz_result"] = gr.Markdown(elem_classes="output-card")

        outs = [UI["quiz_state"], UI["quiz_status"], UI["quiz_submit"], UI["quiz_again"], UI["quiz_result"]] + UI["quiz_radios"]
        ins = [UI["quiz_doc"], UI["quiz_topic"], UI["quiz_num"]]
        e1 = UI["quiz_gen"].click(quiz_generate, ins, outs)
        e2 = UI["quiz_again"].click(quiz_generate, ins, outs)
        e3 = UI["quiz_load"].click(quiz_load_from_chat, None, outs)
        e4 = UI["quiz_submit"].click(quiz_submit, [UI["quiz_state"]] + UI["quiz_radios"],
                                     [UI["quiz_result"]] + UI["quiz_radios"])
        UI["dash_events"].extend([e1, e2, e3, e4])

print("✅ Quiz interface ready")

✅ Quiz interface ready


In [20]:
def research_run(topic):
    topic = (topic or "").strip()
    if not topic:
        return "⚠️ Please enter a research topic.", ""
    res = run_coordinator(topic, None, "Research Agent")
    log_history("research", res["agent"], topic, res["answer"] + "\n\n" + res.get("sources", ""))
    return res["answer"], res.get("sources", "")

def build_research_page():
    with gr.Column(visible=False, elem_classes="page") as col:
        UI["pages"]["Research Assistant"] = col
        gr.Markdown("## 🌐 Research Assistant\nSearches the internet (DuckDuckGo), reads the top pages and writes a sourced summary. "
                    "Online results are kept separate from your uploaded documents.")
        UI["research_topic"] = gr.Textbox(label="Research topic", placeholder="e.g. Latest developments in solar energy")
        with gr.Row():
            UI["research_btn"] = gr.Button("🔎 Search & summarize", variant="primary")
            UI["research_retry"] = gr.Button("🔄 Retry")
        UI["research_out"] = gr.Markdown("Results will appear here.", elem_classes="output-card")
        UI["research_src"] = gr.Markdown()
        e1 = UI["research_btn"].click(research_run, UI["research_topic"], [UI["research_out"], UI["research_src"]])
        e2 = UI["research_retry"].click(research_run, UI["research_topic"], [UI["research_out"], UI["research_src"]])
        UI["dash_events"].extend([e1, e2])

print("✅ Research interface ready")

✅ Research interface ready


In [21]:
def compare_run(a, b):
    if not a or not b:
        return "⚠️ Please select both Document A and Document B.", ""
    if len(DOC_LIBRARY) < 2:
        return "📂 Upload at least two documents in the Document Library first.", ""
    res = run_coordinator(f"Compare {a} and {b}", None, "Document Comparison Agent", [a, b])
    log_history("comparison", res["agent"], f"{a} vs {b}", res["answer"])
    return res["answer"], res.get("sources", "")

def build_compare_page():
    with gr.Column(visible=False, elem_classes="page") as col:
        UI["pages"]["Document Comparison"] = col
        gr.Markdown("## ⚖️ Document Comparison\nPick two uploaded documents. Long documents are condensed in parts before comparing, "
                    "so this can take a minute.")
        with gr.Row():
            UI["cmp_a"] = gr.Dropdown([], value=None, label="Document A")
            UI["cmp_b"] = gr.Dropdown([], value=None, label="Document B")
        UI["cmp_btn"] = gr.Button("⚖️ Compare documents", variant="primary")
        UI["cmp_out"] = gr.Markdown("The comparison will appear here.", elem_classes="output-card")
        UI["cmp_src"] = gr.Markdown()
        ev = UI["cmp_btn"].click(compare_run, [UI["cmp_a"], UI["cmp_b"]], [UI["cmp_out"], UI["cmp_src"]])
        UI["dash_events"].append(ev)

print("✅ Comparison interface ready")

✅ Comparison interface ready


In [22]:
def lib_upload(files):
    return add_documents(files), gr.update(value=None)

def build_library_page():
    with gr.Column(visible=False, elem_classes="page") as col:
        UI["pages"]["Document Library"] = col
        gr.Markdown("## 📚 Document Library\nUpload PDF, DOCX or TXT files. Files are extracted, chunked, embedded and indexed in FAISS automatically.")
        UI["lib_files"] = gr.File(label="Upload documents", file_count="multiple",
                                  file_types=[".pdf", ".docx", ".txt"], type="filepath")
        UI["lib_upload_btn"] = gr.Button("📤 Upload & index", variant="primary")
        UI["lib_status"] = gr.Markdown()
        UI["lib_table"] = gr.Dataframe(value=library_dataframe(), interactive=False, wrap=True, label="Your documents")
        UI["lib_select"] = gr.Dropdown(doc_choices(), value=None, label="Select a document")
        with gr.Row():
            UI["lib_remove"] = gr.Button("🗑️ Remove")
            UI["lib_reindex"] = gr.Button("🔄 Re-index")
            UI["lib_reindex_all"] = gr.Button("🔄 Re-index all")
            UI["lib_view"] = gr.Button("👁️ View extracted text")
            UI["lib_clear"] = gr.Button("🧹 Clear all documents", variant="stop")
        UI["lib_text"] = gr.Textbox(label="Extracted text (first 6000 characters)", lines=12, interactive=False)

        e1 = UI["lib_upload_btn"].click(lib_upload, UI["lib_files"], [UI["lib_status"], UI["lib_files"]])
        e2 = UI["lib_remove"].click(remove_document, UI["lib_select"], UI["lib_status"])
        e3 = UI["lib_reindex"].click(reindex_document, UI["lib_select"], UI["lib_status"])
        e4 = UI["lib_reindex_all"].click(reindex_all, None, UI["lib_status"])
        e5 = UI["lib_clear"].click(clear_library, None, UI["lib_status"])
        UI["lib_view"].click(view_document_text, UI["lib_select"], UI["lib_text"])
        UI["lib_events"].extend([e1, e2, e3, e4, e5])

print("✅ Library interface ready")

✅ Library interface ready


In [23]:
def hist_clear():
    clear_history()
    return history_markdown()

def hist_clear_memory():
    clear_memory()
    return "🧹 Conversation memory cleared. Follow-up questions will start fresh."

def build_history_page():
    with gr.Column(visible=False, elem_classes="page") as col:
        UI["pages"]["Chat History"] = col
        gr.Markdown("## 🕘 Chat History\nQuestions, answers, notes, quiz results and research from this session.")
        with gr.Row():
            UI["hist_refresh"] = gr.Button("🔄 Refresh")
            UI["hist_clear"] = gr.Button("🗑️ Clear history", variant="stop")
            UI["hist_mem"] = gr.Button("🧹 Clear chat memory")
        UI["hist_status"] = gr.Markdown()
        UI["hist_md"] = gr.Markdown(history_markdown())
        UI["hist_refresh"].click(history_markdown, None, UI["hist_md"])
        e = UI["hist_clear"].click(hist_clear, None, UI["hist_md"])
        UI["hist_mem"].click(hist_clear_memory, None, UI["hist_status"])
        UI["dash_events"].append(e)

print("✅ History interface ready")

✅ History interface ready


In [24]:
PROVIDER_PRESETS = {   # examples only: check your provider's docs for current model names
    "OpenAI": ("https://api.openai.com/v1", "gpt-4o-mini"),
    "Groq": ("https://api.groq.com/openai/v1", "llama-3.3-70b-versatile"),
    "OpenRouter": ("https://openrouter.ai/api/v1", "openai/gpt-4o-mini"),
    "Together AI": ("https://api.together.xyz/v1", "meta-llama/Llama-3.3-70B-Instruct-Turbo"),
    "Google Gemini": ("https://generativelanguage.googleapis.com/v1beta/openai/", "gemini-2.0-flash"),
    "Custom": None,
}

def apply_preset(name):
    p = PROVIDER_PRESETS.get(name)
    return (gr.update(value=p[0]), gr.update(value=p[1])) if p else (gr.update(), gr.update())

def settings_save(key, url, model):
    if key and key.strip():
        CONFIG["api_key"] = key.strip()
    if url and url.strip():
        CONFIG["base_url"] = normalize_base_url(url)      # FIX: cleans the URL
    if model and model.strip():
        CONFIG["model"] = model.strip()
    try:
        validate_config()
        msg = f"✅ Settings saved. {key_status()} · Model: `{CONFIG['model']}`"
    except LLMError as e:
        msg = f"⚠️ Saved, but: {e}"
    return msg, ""                      # second value clears the key box

def settings_test(key, url, model):
    msg, _ = settings_save(key, url, model)
    ok, result = test_connection()
    return result, ""

def apply_chunk_settings(size, overlap, k):
    CONFIG["chunk_size"] = int(size)
    CONFIG["chunk_overlap"] = min(int(overlap), int(size) // 2)
    CONFIG["top_k"] = int(k)
    if not DOC_LIBRARY:
        return "✅ RAG settings saved. They will be used for the next uploaded documents."
    return "✅ RAG settings saved. " + reindex_all()

def build_settings_page():
    with gr.Column(visible=False, elem_classes="page") as col:
        UI["pages"]["Settings"] = col
        gr.Markdown("## ⚙️ Settings\nUse any **OpenAI-compatible** provider. Your API key is kept only in memory for this session "
                    "and is never displayed.")
        UI["set_preset"] = gr.Dropdown(list(PROVIDER_PRESETS), value="Custom", label="Provider preset (optional)")
        UI["set_key"] = gr.Textbox(label="API Key", type="password", placeholder="Paste your key (leave empty to keep the current key)")
        UI["set_url"] = gr.Textbox(label="Base URL", value=CONFIG["base_url"])
        UI["set_model"] = gr.Textbox(label="Model name", value=CONFIG["model"])
        with gr.Row():
            UI["set_save"] = gr.Button("💾 Save settings", variant="primary")
            UI["set_test"] = gr.Button("🔌 Test connection")
        UI["set_status"] = gr.Markdown(key_status())

        gr.Markdown("### 🧩 RAG settings")
        with gr.Row():
            UI["set_chunk"] = gr.Slider(300, 2000, value=CONFIG["chunk_size"], step=50, label="Chunk size (characters)")
            UI["set_overlap"] = gr.Slider(0, 500, value=CONFIG["chunk_overlap"], step=25, label="Chunk overlap")
            UI["set_topk"] = gr.Slider(2, 10, value=CONFIG["top_k"], step=1, label="Chunks retrieved (top-k)")
        UI["set_apply"] = gr.Button("♻️ Apply & re-index documents")
        UI["set_rag_status"] = gr.Markdown()
        gr.Markdown("> ℹ️ Chat memory, history and documents are temporary and reset when the Colab runtime disconnects.")

        UI["set_preset"].change(apply_preset, UI["set_preset"], [UI["set_url"], UI["set_model"]])
        ins = [UI["set_key"], UI["set_url"], UI["set_model"]]
        UI["set_save"].click(settings_save, ins, [UI["set_status"], UI["set_key"]])
        UI["set_test"].click(settings_test, ins, [UI["set_status"], UI["set_key"]])
        ev = UI["set_apply"].click(apply_chunk_settings, [UI["set_chunk"], UI["set_overlap"], UI["set_topk"]], UI["set_rag_status"])
        UI["lib_events"].append(ev)

print("✅ Settings interface ready")

✅ Settings interface ready


In [25]:
THEME = gr.themes.Base(
    primary_hue="violet", secondary_hue="blue", neutral_hue="slate",
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui", "sans-serif"],
).set(
    body_background_fill_dark="#0b1020",
    background_fill_primary_dark="#0f172a",
    background_fill_secondary_dark="#111a33",
    block_background_fill_dark="#111a33",
    block_border_color_dark="#243056",
    input_background_fill_dark="#0b1226",
    button_primary_background_fill="linear-gradient(135deg,#7c3aed,#2563eb)",
    button_primary_background_fill_hover="linear-gradient(135deg,#8b5cf6,#3b82f6)",
    button_primary_text_color="white",
)

DARK_JS = "() => { document.body.classList.add('dark'); }"

BRAND_HTML = f"""
<div class="brand"><div class="brand-logo">🧠</div>
<div><div class="brand-title">AI Personal<br>Knowledge Assistant</div>
<div class="brand-sub">Multi-agent RAG · Colab</div></div></div>
"""

SIDEBAR_NOTE = """<div class="side-note">Documents, memory and history live in this Colab session only.
The public link works while the runtime is running.</div>"""

CUSTOM_CSS = """
:root { --navy:#0b1020; --card:#111a33; --border:#243056; --purple:#7c3aed; --blue:#2563eb; --text:#e5e7eb; --muted:#9ca3af; }
.gradio-container {
  max-width: 1500px !important; color: var(--text);
  background: radial-gradient(1100px 600px at 8% -10%, rgba(124,58,237,.25), transparent 60%),
              radial-gradient(900px 500px at 100% 0%, rgba(37,99,235,.20), transparent 55%), #0b1020 !important;
}
footer { display: none !important; }

/* sidebar */
.sidebar { background: linear-gradient(180deg,#111a33,#0d1429) !important; border: 1px solid var(--border);
  border-radius: 20px; padding: 16px !important; position: sticky; top: 12px; align-self: flex-start; }
.brand { display:flex; align-items:center; gap:12px; margin-bottom:14px; }
.brand-logo { font-size:30px; width:52px; height:52px; display:flex; align-items:center; justify-content:center;
  border-radius:16px; background: linear-gradient(135deg,#7c3aed,#2563eb); box-shadow: 0 6px 20px rgba(124,58,237,.45); }
.brand-title { font-weight:800; font-size:15px; line-height:1.2; color:#fff; }
.brand-sub { font-size:11px; color: var(--muted); margin-top:2px; }
.nav-btn { justify-content:flex-start !important; text-align:left !important; border-radius:12px !important;
  margin-bottom:6px !important; font-weight:600 !important; transition: all .2s ease; }
.nav-btn:hover { transform: translateX(4px); }
button.nav-btn.secondary { background: transparent !important; border: 1px solid transparent !important; color:#cbd5e1 !important; }
button.nav-btn.secondary:hover { background: rgba(124,58,237,.15) !important; border-color: var(--border) !important; }
button.nav-btn.primary { background: linear-gradient(135deg,#7c3aed,#2563eb) !important; color:#fff !important; }
.side-note { font-size:11px; color: var(--muted); margin-top:14px; line-height:1.5; }

/* pages */
.page { animation: fadeIn .35s ease; }
@keyframes fadeIn { from { opacity:0; transform: translateY(8px); } to { opacity:1; transform:none; } }
.main-area h2 { background: linear-gradient(90deg,#c4b5fd,#93c5fd); -webkit-background-clip:text; background-clip:text;
  -webkit-text-fill-color: transparent; font-weight:800; }

/* dashboard */
.hero { display:flex; gap:20px; align-items:center; padding:26px; border-radius:22px; border:1px solid var(--border);
  background: linear-gradient(135deg, rgba(124,58,237,.30), rgba(37,99,235,.22)); margin-bottom:16px; }
.hero-logo { font-size:54px; width:92px; height:92px; flex-shrink:0; display:flex; align-items:center; justify-content:center;
  border-radius:24px; background: linear-gradient(135deg,#7c3aed,#2563eb); box-shadow: 0 10px 30px rgba(124,58,237,.5); }
.hero h1 { margin:0 0 6px 0; font-size:28px; font-weight:800; color:#fff; }
.hero p { margin:0 0 10px 0; color:#d1d5db; line-height:1.55; }
.chips { display:flex; flex-wrap:wrap; gap:8px; }
.chip { padding:4px 12px; border-radius:999px; font-size:12px; color:#e0e7ff; background: rgba(99,102,241,.25); border:1px solid rgba(129,140,248,.4); }
.stat-grid { display:grid; grid-template-columns: repeat(auto-fit, minmax(160px, 1fr)); gap:14px; margin-bottom:10px; }
.stat-card { padding:18px; border-radius:18px; background: var(--card); border:1px solid var(--border); text-align:center; transition: transform .2s, box-shadow .2s; }
.stat-card:hover { transform: translateY(-4px); box-shadow: 0 10px 28px rgba(124,58,237,.25); }
.stat-icon { font-size:26px; }
.stat-value { font-size:32px; font-weight:800; color:#fff; }
.stat-label { font-size:12px; color: var(--muted); }
.qa-btn { border-radius:16px !important; min-height:64px; font-weight:700 !important; background: var(--card) !important;
  border:1px solid var(--border) !important; transition: all .2s ease; }
.qa-btn:hover { background: linear-gradient(135deg, rgba(124,58,237,.35), rgba(37,99,235,.3)) !important; transform: translateY(-3px); }

/* chat */
.message.user { background: linear-gradient(135deg,#7c3aed,#2563eb) !important; color:#fff !important; border-radius:18px 18px 4px 18px !important; }
.message.bot  { background:#162044 !important; border:1px solid var(--border) !important; border-radius:18px 18px 18px 4px !important; }
.chat-box { border-radius:18px !important; }

/* cards / inputs / buttons */
.output-card { padding:16px; border-radius:16px; border:1px solid var(--border); background: var(--card); }
.quiz-q { border-radius:14px !important; border:1px solid var(--border) !important; padding:10px !important; }
button.primary { border-radius:12px !important; font-weight:700 !important; transition: transform .15s, box-shadow .15s; }
button.primary:hover { transform: translateY(-2px); box-shadow: 0 8px 22px rgba(124,58,237,.45); }
textarea, input, .gr-box { border-radius:12px !important; }

/* responsive */
@media (max-width: 900px) {
  .sidebar { position: static; }
  .hero { flex-direction: column; text-align: center; }
  .hero h1 { font-size: 22px; }
}
"""

print("✅ Theme and CSS ready")

✅ Theme and CSS ready


In [26]:
PAGE_ORDER = [("Dashboard", "🏠"), ("AI Knowledge Chat", "💬"), ("Study Assistant", "🎓"),
              ("Quiz Center", "📝"), ("Research Assistant", "🌐"), ("Document Comparison", "⚖️"),
              ("Document Library", "📚"), ("Chat History", "🕘"), ("Settings", "⚙️")]

def make_show_page(target):
    def _show():
        return ([gr.update(visible=(n == target)) for n, _ in PAGE_ORDER] +
                [gr.update(variant="primary" if n == target else "secondary") for n, _ in PAGE_ORDER])
    return _show

def refresh_docs():
    names = doc_choices()
    return (library_dataframe(),
            gr.update(choices=names, value=None),                    # library selector
            gr.update(choices=[NO_DOC] + names, value=NO_DOC),       # study document
            gr.update(choices=[NO_DOC] + names, value=NO_DOC),       # quiz document
            gr.update(choices=names, value=None),                    # compare A
            gr.update(choices=names, value=None),                    # compare B
            gr.update(choices=[ALL_DOCS] + names, value=ALL_DOCS))   # chat filter

def build_app():
    reset_ui()
    with gr.Blocks(title=APP_TITLE, theme=THEME, css=CUSTOM_CSS, js=DARK_JS) as app:
        with gr.Row(equal_height=False):
            with gr.Column(scale=1, min_width=250, elem_classes="sidebar"):
                gr.HTML(BRAND_HTML)
                nav = {}
                for name, icon in PAGE_ORDER:
                    nav[name] = gr.Button(f"{icon}  {name}", variant="primary" if name == "Dashboard" else "secondary",
                                          elem_classes="nav-btn")
                gr.HTML(SIDEBAR_NOTE)
            with gr.Column(scale=5, elem_classes="main-area"):
                build_dashboard_page()
                build_chat_page()
                build_study_page()
                build_quiz_page()
                build_research_page()
                build_compare_page()
                build_library_page()
                build_history_page()
                build_settings_page()

        page_cols = [UI["pages"][n] for n, _ in PAGE_ORDER]
        nav_btns = [nav[n] for n, _ in PAGE_ORDER]
        dash_out = [UI["dash_html"], UI["dash_plot"], UI["dash_activity"]]
        docs_out = [UI["lib_table"], UI["lib_select"], UI["study_doc"], UI["quiz_doc"],
                    UI["cmp_a"], UI["cmp_b"], UI["chat_doc"]]

        # sidebar navigation
        for name, _ in PAGE_ORDER:
            ev = nav[name].click(make_show_page(name), None, page_cols + nav_btns)
            if name == "Dashboard":
                ev.then(refresh_dashboard, None, dash_out)
            if name == "Chat History":
                ev.then(history_markdown, None, UI["hist_md"])

        # dashboard quick-action cards
        quick = {"qa_chat": "AI Knowledge Chat", "qa_upload": "Document Library", "qa_study": "Study Assistant",
                 "qa_quiz": "Quiz Center", "qa_research": "Research Assistant", "qa_compare": "Document Comparison"}
        for key, target in quick.items():
            UI[key].click(make_show_page(target), None, page_cols + nav_btns)

        # library/settings changes -> refresh dropdowns, table and dashboard
        for ev in UI["lib_events"]:
            ev.then(refresh_docs, None, docs_out).then(refresh_dashboard, None, dash_out)
        # other actions -> refresh dashboard statistics
        for ev in UI["dash_events"]:
            ev.then(refresh_dashboard, None, dash_out)

        # when the page is (re)loaded
        app.load(refresh_docs, None, docs_out)
        app.load(refresh_dashboard, None, dash_out)
        app.load(history_markdown, None, UI["hist_md"])
    return app

demo = build_app()
print("✅ Website assembled. Pages:", ", ".join(n for n, _ in PAGE_ORDER))

/tmp/ipykernel_988/3134398170.py:23: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title=APP_TITLE, theme=THEME, css=CUSTOM_CSS, js=DARK_JS) as app:
/tmp/ipykernel_988/3134398170.py:23: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title=APP_TITLE, theme=THEME, css=CUSTOM_CSS, js=DARK_JS) as app:
/tmp/ipykernel_988/3134398170.py:23: DeprecationWarning: The 'js' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'js' to Blocks.launch() instead.
  with gr.Blocks(title=APP_TITLE, theme=THEME, css=CUSTOM_CSS, js=DARK_JS) as app:
/tmp/ipykernel_988/248518398.py:43: DeprecationWarning:

The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.

/tmp/ipy

✅ Website assembled. Pages: Dashboard, AI Knowledge Chat, Study Assistant, Quiz Center, Research Assistant, Document Comparison, Document Library, Chat History, Settings


In [27]:
import tempfile

TEST_DIR = tempfile.mkdtemp()
ML_TEXT = """Machine Learning Notes
Machine learning is a branch of artificial intelligence where computers learn patterns from data instead of being explicitly programmed.
Supervised learning uses labeled data to train models for classification and regression. Examples are spam detection and house price prediction.
Unsupervised learning finds hidden structure in unlabeled data. Clustering and dimensionality reduction are common techniques.
Reinforcement learning trains an agent through rewards and penalties while it interacts with an environment.
Overfitting happens when a model memorizes the training data and performs poorly on new data. Regularization, cross-validation and more data help prevent overfitting.
Evaluation metrics include accuracy, precision, recall and the F1 score."""
DL_TEXT = """Deep Learning Notes
Deep learning is a subfield of machine learning that uses neural networks with many layers.
Convolutional neural networks (CNNs) are used for images. Recurrent neural networks (RNNs) and transformers are used for sequences such as text.
Backpropagation computes gradients so that weights can be updated with gradient descent.
Deep learning needs large datasets and powerful hardware such as GPUs.
Dropout and batch normalization are techniques that help deep networks train better and avoid overfitting."""

def _check(name, cond, detail=""):
    print(("✅ PASS" if cond else "❌ FAIL"), "-", name, detail)
    return bool(cond)

def _need_key():
    if not CONFIG["api_key"]:
        print("⏭️  SKIPPED (no API key set; add one in Settings or Cell 4)")
        return False
    return True

def _setup_samples():
    pa, pb = os.path.join(TEST_DIR, "ml_notes.txt"), os.path.join(TEST_DIR, "dl_notes.txt")
    open(pa, "w").write(ML_TEXT)
    open(pb, "w").write(DL_TEXT)
    add_documents([pa, pb])

def test_extraction():
    print("\n=== Extraction tests (PDF / DOCX / TXT / unsupported) ===")
    pdf_path = os.path.join(TEST_DIR, "sample.pdf")
    d = fitz.open(); p = d.new_page()
    p.insert_text((72, 100), "Gradient descent minimizes a loss function by updating weights step by step.")
    d.save(pdf_path); d.close()
    r = extract_document(pdf_path)
    _check("PDF text + page number", r["pages_data"][0]["page"] == 1 and "Gradient" in r["pages_data"][0]["text"])
    docx_path = os.path.join(TEST_DIR, "sample.docx")
    dd = docx.Document(); dd.add_paragraph("Python is a popular programming language."); dd.save(docx_path)
    _check("DOCX text", "Python" in extract_document(docx_path)["pages_data"][0]["text"])
    _check("TXT text", "Machine learning" in extract_document(os.path.join(TEST_DIR, "ml_notes.txt"))["pages_data"][0]["text"])
    try:
        extract_document("virus.exe"); _check("Unsupported file rejected", False)
    except ExtractionError:
        _check("Unsupported file rejected", True)

def test_rag():
    print("\n=== RAG / FAISS retrieval test ===")
    hits = retrieve("What is overfitting?", k=3)
    _check("FAISS returns chunks", len(hits) > 0)
    _check("Top hit from correct document", hits and hits[0]["doc"] == "ml_notes.txt", f"(got {hits[0]['doc'] if hits else None})")
    _check("Citation has no invented page", hits and "page unavailable" in citation(hits[0]), f"-> {citation(hits[0]) if hits else ''}")

def test_quiz_grading():
    print("\n=== Quiz scoring test (no LLM needed) ===")
    qs = [{"question": f"Q{i}", "options": ["a", "b", "c", "d"], "answer": a, "explanation": "x"} for i, a in enumerate("ABC")]
    res = grade_quiz(qs, ["A", "C", None])
    _check("Score calculated", res["score"] == 1 and res["total"] == 3 and res["percentage"] == 33.3, f"-> {res['score']}/{res['total']} ({res['percentage']}%)")

def test_routing_rules():
    print("\n=== Keyword routing rules (fallback router) ===")
    cases = {"Compare the two documents": "comparison", "Create a 5 question quiz": "quiz",
             "Search the web for ISRO news": "research", "Explain neural networks simply": "study"}
    for q, exp in cases.items():
        _check(f"'{q}' -> {exp}", heuristic_route(q, True)[0] == exp)

def test_knowledge_agent():
    print("\n=== Knowledge Agent ===")
    if not _need_key(): return
    r = run_coordinator("What is overfitting and how can it be prevented?", forced_agent="Knowledge Agent")
    _check("Answer generated", len(r["answer"]) > 30 and not r["answer"].startswith("⚠️"), "\n" + r["answer"][:300])
    _check("Source citation appears", "[Source:" in r["answer"] or "ml_notes.txt" in r["sources"])

def test_study_agent():
    print("\n=== Study Agent ===")
    if not _need_key(): return
    r = run_coordinator("supervised learning", forced_agent="Study Agent", doc_names=["ml_notes.txt"],
                        study_task="Generate notes", study_style="Bullet points")
    _check("Notes generated", len(r["answer"]) > 50 and not r["answer"].startswith("⚠️"), "\n" + r["answer"][:300])
    _check("Context retrieved from document", "ml_notes.txt" in r["sources"])

def test_quiz_agent():
    print("\n=== Quiz Agent (5 MCQs) ===")
    if not _need_key(): return
    qs = generate_quiz("ml_notes.txt", "", 5)
    _check("5 questions generated", len(qs) == 5, f"(got {len(qs)})")
    _check("Four options + stored answers", all(len(q["options"]) == 4 and q["answer"] in "ABCD" for q in qs))
    res = grade_quiz(qs, [q["answer"] for q in qs])
    _check("Perfect answers score 100%", res["percentage"] == 100.0)
    _check("Explanations available after submit", "Explanation" in res["details"])

def test_research_agent():
    print("\n=== Research Agent (real web search) ===")
    if not _need_key(): return
    r = run_coordinator("latest developments in solar energy", forced_agent="Research Agent")
    ok = "http" in r.get("sources", "")
    _check("Real search performed, clickable URLs", ok, "\n" + (r.get("sources", "") or r["answer"])[:400])

def test_comparison_agent():
    print("\n=== Comparison Agent ===")
    if not _need_key(): return
    r = comparison_agent("ml_notes.txt", "dl_notes.txt")
    _check("Similarities section", "Similarities" in r["answer"])
    _check("Differences section", "Differences" in r["answer"])
    _check("Final summary", "Final Summary" in r["answer"])

def test_coordinator():
    print("\n=== Coordinator routing (Auto Agent) ===")
    if not _need_key(): return
    cases = [("What does my document say about reinforcement learning?", "Knowledge Agent"),
             ("Explain backpropagation in simple words and give me a 10-mark answer", "Study Agent"),
             ("Create a 5-question quiz on machine learning", "Quiz Agent"),
             ("Search the web for latest news about ISRO", "Research Agent"),
             ("Compare ml_notes.txt and dl_notes.txt", "Document Comparison Agent")]
    for q, exp in cases:
        r = run_coordinator(q)
        _check(f"'{q[:50]}…' -> {exp}", r["agent"] == exp, f"(routed to {r['agent']}: {r.get('reason', '')})")

def run_all_tests():
    _setup_samples()
    for t in (test_extraction, test_rag, test_quiz_grading, test_routing_rules, test_knowledge_agent,
              test_study_agent, test_quiz_agent, test_research_agent, test_comparison_agent, test_coordinator):
        try:
            t()
        except Exception as e:
            print("❌ FAIL -", t.__name__, "crashed:", e)
    for n in ("ml_notes.txt", "dl_notes.txt"):          # remove test files from your library
        DOC_LIBRARY.pop(n, None)
    try:
        rebuild_index()
    except Exception:
        STORE.reset()
    LAST_QUIZ["questions"] = []
    print("\n🏁 Tests finished. Sample documents were removed from your library.")

print("✅ Test functions ready. Run:  run_all_tests()")

✅ Test functions ready. Run:  run_all_tests()


In [28]:
run_all_tests()


=== Extraction tests (PDF / DOCX / TXT / unsupported) ===
✅ PASS - PDF text + page number 
✅ PASS - DOCX text 
✅ PASS - TXT text 
✅ PASS - Unsupported file rejected 

=== RAG / FAISS retrieval test ===
✅ PASS - FAISS returns chunks 
✅ PASS - Top hit from correct document (got ml_notes.txt)
✅ PASS - Citation has no invented page -> [Source: ml_notes.txt, page unavailable]

=== Quiz scoring test (no LLM needed) ===
✅ PASS - Score calculated -> 1/3 (33.3%)

=== Keyword routing rules (fallback router) ===
✅ PASS - 'Compare the two documents' -> comparison 
✅ PASS - 'Create a 5 question quiz' -> quiz 
✅ PASS - 'Search the web for ISRO news' -> research 
✅ PASS - 'Explain neural networks simply' -> study 

=== Knowledge Agent ===
✅ PASS - Answer generated 
Overfitting happens when a model memorizes the training data and performs poorly on new data. Regularization, cross-validation, and more data help prevent overfitting [Source: ml_notes.txt, page unavailable].
✅ PASS - Source citation appe

In [ ]:
try:
    demo.close()                      # closes an older running copy if you re-run this cell
except Exception:
    pass

demo.queue(max_size=32)               # allows long AI tasks without blocking
app, local_url, share_url = demo.launch(share=True, show_error=True, debug=False, prevent_thread_lock=True)

print("=" * 70)
print("🌐 Local URL :", local_url)
print("🌍 PUBLIC URL:", share_url if share_url else "(share link could not be created; check your internet)")
print("=" * 70)
print("Open the PUBLIC URL in your browser. It works while this Colab runtime is running,")
print("and the temporary share link expires when the runtime stops.")
print("Keep this cell running (click ■ to stop the website).")

try:
    demo.block_thread()
except KeyboardInterrupt:
    demo.close()

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b436b0776cf2253eca.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


🌐 Local URL : http://127.0.0.1:7860/
🌍 PUBLIC URL: https://b436b0776cf2253eca.gradio.live
Open the PUBLIC URL in your browser. It works while this Colab runtime is running,
and the temporary share link expires when the runtime stops.
Keep this cell running (click ■ to stop the website).
